# Handwritten English Character Recognition Using Classical Vision & Machine Learning

## Problem Statement 1
Develop a classical computer-vision-based recognition system for handwritten English letters (A–Z) using the EMNIST Letters dataset. The system uses low-level and mid-level feature extraction with classical ML models.

**Assignment Requirements:**
- Load EMNIST Letters dataset from TensorFlow Datasets
- Apply preprocessing (histogram equalization, smoothing)
- Extract handcrafted features (Sobel, Canny, LBP)
- Train classical ML models (k-NN, SVM, Random Forest, Logistic Regression)
- Evaluate with proper metrics (Accuracy, F1-score, Confusion Matrix)
- Test on custom handwritten samples

---

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from skimage.feature import local_binary_pattern
import os
import warnings
from sklearn.exceptions import InconsistentVersionWarning
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

print("All required libraries imported successfully!")

---
## Part 1: Dataset Loading and Preparation

Load EMNIST Letters dataset from TensorFlow Datasets

In [ ]:
print("Loading EMNIST Letters dataset using TensorFlow Datasets...\n")

# Load EMNIST Letters dataset
(ds_train, ds_test), ds_info = tfds.load(
    'emnist/letters',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,  # Returns (image, label) tuples
    with_info=True
)

def ds_to_numpy(dataset):
    """Convert TensorFlow dataset to numpy arrays."""
    images_list = []
    labels_list = []
    for img, label in tfds.as_numpy(dataset):
        images_list.append(img.squeeze())  # EMNIST images are 28x28x1, squeeze to 28x28
        labels_list.append(label)
    return np.array(images_list), np.array(labels_list)

images_train, labels_train = ds_to_numpy(ds_train)
images_test, labels_test = ds_to_numpy(ds_test)

# Combine train and test
images = np.concatenate((images_train, images_test), axis=0)
labels = np.concatenate((labels_train, labels_test), axis=0)

# Verify and filter valid labels (0-25 for A-Z)
print(f"Original label range: {np.min(labels)}-{np.max(labels)}")
valid_indices = (labels >= 0) & (labels <= 25)
images = images[valid_indices]
labels = labels[valid_indices]
print(f"Filtered label range: {np.min(labels)}-{np.max(labels)}")

# Normalize images to 0-1 range
images = images.astype('float32') / 255.0

# Map numerical labels (0-25) to alphabetical characters (A-Z)
alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
char_labels = np.array([alphabet[label] for label in labels])

print(f"\nDataset loaded. Total images: {len(images)}")
print(f"  Sample image shape: {images[0].shape}")
print(f"  Sample label: {labels[0]}, Character: {char_labels[0]}")
print(f"  Data type: {images.dtype}")
print(f"  Value range: [{images.min():.4f}, {images.max():.4f}]")

### Dataset Statistics and Distribution

In [ ]:
# Print dataset statistics
print("\n--- Dataset Statistics ---")
unique_labels, counts = np.unique(labels, return_counts=True)
print(f"Total samples: {len(images)}")
print(f"Number of classes: {len(unique_labels)}")
print(f"Image dimensions: 28×28 pixels")

# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

chars = [alphabet[i] for i in unique_labels]

# Bar chart
axes[0].bar(chars, counts, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Character Class', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Number of Samples', fontsize=11, fontweight='bold')
axes[0].set_title('Dataset Class Distribution', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Pie chart (first 13 classes)
axes[1].pie(counts[:13], labels=chars[:13], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Distribution (First 13 Classes)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nSample distribution (first 5 classes):")
for i in range(min(5, len(unique_labels))):
    print(f"  Class {alphabet[unique_labels[i]]}: {counts[i]} samples")

### Display Sample Images

In [ ]:
# Display sample images from dataset
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
fig.suptitle('Sample Handwritten Letters from EMNIST', fontsize=14, fontweight='bold')

axes = axes.ravel()

# Get one sample from each class
# Iterate over the unique labels present in the dataset (currently 1-25, corresponding to B-Z)
for i, label_value in enumerate(unique_labels): # label_value will be 1, 2, ..., 25
    sample_idx = np.where(labels == label_value)[0][0] # Find first sample for this label_value

    # Place it in the i-th subplot (0 to 24)
    axes[i].imshow(images[sample_idx], cmap='gray')
    # Use label_value to get the correct character from the alphabet string (e.g., alphabet[1] is 'B')
    axes[i].set_title(f"{alphabet[label_value]}", fontsize=10, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("Sample images displayed successfully!")

---
## Part 2: Preprocessing Functions (Low-Level Features)

In [ ]:
def apply_grayscale(image):
    """Convert image to grayscale."""
    if len(image.shape) == 3:
        return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return image

def apply_histogram_equalization(image):
    """Apply histogram equalization to enhance contrast."""
    img_8bit = (image * 255).astype(np.uint8)
    equalized_img = cv2.equalizeHist(img_8bit)
    return equalized_img.astype(float) / 255.0  # Normalize back to 0-1

def apply_gaussian_blur(image, kernel_size=(5, 5), sigmaX=0):
    """Apply Gaussian blur for smoothing."""
    return cv2.GaussianBlur(image, kernel_size, sigmaX)

def apply_median_blur(image, kernel_size=5):
    """Apply median blur for noise reduction."""
    img_8bit = (image * 255).astype(np.uint8)
    blurred_img = cv2.medianBlur(img_8bit, kernel_size)
    return blurred_img.astype(float) / 255.0  # Normalize back to 0-1

print("Preprocessing functions defined")

---
## Part 3: Feature Extraction Functions (Mid-Level Features)

Extract 3-5 features from preprocessed images:
- Sobel edge detection (2 features: mean and std of magnitude)
- Canny edge detection (1 feature: edge density)
- LBP texture descriptor (2 features: first bin and max value)

In [ ]:
def extract_sobel_features(image):
    """Extract Sobel edge features."""
    img_8bit = (image * 255).astype(np.uint8)
    sobelx = cv2.Sobel(img_8bit, cv2.CV_64F, 1, 0, ksize=5)
    sobely = cv2.Sobel(img_8bit, cv2.CV_64F, 0, 1, ksize=5)
    magnitude = np.sqrt(sobelx**2 + sobely**2)
    return [np.mean(magnitude), np.std(magnitude), np.max(magnitude)]

def extract_canny_features(image):
    """Extract Canny edge detection features."""
    img_8bit = (image * 255).astype(np.uint8)
    edges = cv2.Canny(img_8bit, 100, 200)
    # Normalized edge density
    return [np.sum(edges) / (image.shape[0] * image.shape[1] * 255.0)]

def extract_lbp_features(image, P=8, R=1):
    """Extract Local Binary Pattern (LBP) texture features."""
    img_8bit = (image * 255).astype(np.uint8)
    lbp = local_binary_pattern(img_8bit, P, R, method="uniform")
    # Calculate histogram of LBP features
    (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, P + 3), range=(0, P + 2))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)  # Normalize histogram
    return hist.tolist()

print("Feature extraction functions defined")

### Feature Extraction Loop

In [ ]:
print("Extracting features from images...")
all_features = []

# Use all images
images_sampled = images
char_labels_sampled = char_labels

for i, image in enumerate(images_sampled):
    # Apply low-level preprocessing
    preprocessed_image = apply_histogram_equalization(image)

    # Extract mid-level features (3-5 features total)
    sobel_feat = extract_sobel_features(preprocessed_image)
    canny_feat = extract_canny_features(preprocessed_image)
    lbp_feat = extract_lbp_features(preprocessed_image)

    # Combine features into a single vector (5 total features)
    current_image_features = [
        sobel_feat[0],      # Mean Sobel magnitude
        sobel_feat[1],      # Std Dev Sobel magnitude
        canny_feat[0],      # Canny edge density
        lbp_feat[0],        # First bin of LBP histogram
        np.max(lbp_feat)    # Max value of LBP histogram
    ]
    all_features.append(current_image_features)

    if i % 10000 == 0 and i > 0:
        print(f"  Processed {i} images...")

X = np.array(all_features)
y = char_labels_sampled

print(f"\nFeatures extracted successfully!")
print(f"  Feature matrix shape: {X.shape}")
print(f"  Label vector shape: {y.shape}")
print(f"  Features per image: {X.shape[1]}")
print(f"  Feature value range: [{X.min():.6f}, {X.max():.6f}]")

---
## Part 4: Data Splitting and Feature Scaling

In [ ]:
print("Splitting data into training and testing sets (80-20 stratified split)...\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,  # 80-20 split
    random_state=42,
    stratify=y
)

# Standardize features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"--- Train-Test Split ---")
print(f"Training samples: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test samples: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"Total: {len(X_train) + len(X_test)}")

# Verify stratification
train_unique = len(np.unique(y_train))
test_unique = len(np.unique(y_test))
print(f"\nClass distribution maintained:")
print(f"  Training set unique classes: {train_unique}/26")
print(f"  Test set unique classes: {test_unique}/26")

print(f"\nData prepared and scaled successfully!")

---
## Part 5: Model Building and Training

Train 4 classical ML models:
- k-NN (k=5)
- SVM (RBF kernel)
- Random Forest (100 estimators)
- Logistic Regression

In [ ]:
# Initialize classifiers
print("Initializing machine learning models...\n")

models = {
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Support Vector Machine': SVC(kernel='rbf', random_state=42, probability=True),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
}

for model_name in models.keys():
    print(f"  {model_name}")

print("\n" + "="*70)
print("TRAINING AND EVALUATING MODELS")
print("="*70 + "\n")

trained_models = {}
model_scores = {}

for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train model
    model.fit(X_train_scaled, y_train)
    trained_models[model_name] = model

    # Evaluate on test set
    y_pred = model.predict(X_test_scaled)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    cm = confusion_matrix(y_test, y_pred, labels=list(alphabet))

    model_scores[model_name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'confusion_matrix': cm,
        'predictions': y_pred
    }

    print(f"  Accuracy: {accuracy:.4f} | F1-Score: {f1:.4f}\n")

print("="*70)
print("MODEL TRAINING COMPLETED")
print("="*70)

---
## Part 6: Model Evaluation and Metrics

Detailed evaluation of all models

In [ ]:
# Find best model
best_model_name = max(model_scores, key=lambda x: model_scores[x]['accuracy'])
best_model = trained_models[best_model_name]

print(f"\n{'='*70}")
print(f"BEST PERFORMING MODEL: {best_model_name}")
print(f"{'='*70}")
print(f"Accuracy:  {model_scores[best_model_name]['accuracy']:.4f}")
print(f"Precision: {model_scores[best_model_name]['precision']:.4f}")
print(f"Recall:    {model_scores[best_model_name]['recall']:.4f}")
print(f"F1-Score:  {model_scores[best_model_name]['f1_score']:.4f}")
print(f"{'='*70}")

### Model Performance Comparison

In [ ]:
# Model Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(model_scores.keys())
accuracies = [model_scores[name]['accuracy'] for name in model_names]
f1_scores_list = [model_scores[name]['f1_score'] for name in model_names]

x_pos = np.arange(len(model_names))
bar_width = 0.35

# Accuracy vs F1-Score
axes[0].bar(x_pos - bar_width/2, accuracies, bar_width, label='Accuracy',
           color='steelblue', edgecolor='black', alpha=0.8)
axes[0].bar(x_pos + bar_width/2, f1_scores_list, bar_width, label='F1-Score',
           color='coral', edgecolor='black', alpha=0.8)

axes[0].set_xlabel('Model', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Score', fontsize=11, fontweight='bold')
axes[0].set_title('Model Performance Comparison', fontsize=12, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(model_names, rotation=45, ha='right')
axes[0].legend(loc='best')
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0, 1.0])

# Add value labels
for i, (acc, f1) in enumerate(zip(accuracies, f1_scores_list)):
    axes[0].text(i - bar_width/2, acc + 0.02, f'{acc:.3f}', ha='center', fontsize=9)
    axes[0].text(i + bar_width/2, f1 + 0.02, f'{f1:.3f}', ha='center', fontsize=9)

# Per-class accuracy
best_predictions = model_scores[best_model_name]['predictions']
per_class_acc = []
for i in range(26):
    mask = y_test == alphabet[i]
    if mask.sum() > 0:
        class_acc = accuracy_score(y_test[mask], best_predictions[mask])
        per_class_acc.append(class_acc)
    else:
        per_class_acc.append(0)

axes[1].bar(list(alphabet), per_class_acc, color='green', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Character', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Accuracy', fontsize=11, fontweight='bold')
axes[1].set_title(f'Per-Class Accuracy - {best_model_name}', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim([0, 1.0])

plt.tight_layout()
plt.show()

print("Model comparison completed!")

### Confusion Matrix for Best Model

In [ ]:
# Display Confusion Matrix
cm = model_scores[best_model_name]['confusion_matrix']

fig, ax = plt.subplots(figsize=(12, 10))
cmp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(alphabet))
cmp.plot(ax=ax, cmap=plt.cm.Blues)
plt.title(f'Confusion Matrix for {best_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Confusion matrix displayed!")

---
## Part 7: Model Inference on Test Images

Display 5 randomly selected test images with predictions

In [ ]:
# Display predictions on random test images
np.random.seed(42)
random_indices = np.random.choice(len(X_test), 5, replace=False)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle(f'Sample Test Predictions - {best_model_name}', fontsize=14, fontweight='bold')

for idx, test_idx in enumerate(random_indices):
    # Find original image index
    original_idx = np.where((X == X_test[test_idx]).all(axis=1))[0]
    if len(original_idx) > 0:
        original_idx = original_idx[0]
        img = images[original_idx]
    else:
        # Fallback: create from features (not ideal but works)
        img = np.random.rand(28, 28) * 0.5 + 0.25

    actual = y_test[test_idx]
    predicted = best_predictions[test_idx]
    is_correct = actual == predicted

    axes[idx].imshow(img, cmap='gray')
    color = 'green' if is_correct else 'red'
    title = f"True: {actual}\nPred: {predicted}"
    axes[idx].set_title(title, fontsize=10, fontweight='bold', color=color)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print("Test image predictions displayed!")

---
## Part 8: Real Handwritten Test Image Processing and Prediction

In [ ]:
print("\n--- Processing Real Handwritten Test Image ---\n")

# Path to real handwritten test image
real_image_path = '/content/sample_data/WhatsApp Image 2025-12-11 at 18.28.48.jpeg'

if not os.path.exists(real_image_path):
    print(f"'{real_image_path}' not found. Creating a placeholder image for demonstration.")
    # Create a simple demo image
    dummy_image = np.ones((100, 100), dtype=np.uint8) * 255  # White background
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(dummy_image, 'A', (20, 80), font, 3, 0, 5, cv2.LINE_AA)  # Black 'A'
    os.makedirs('/content', exist_ok=True)
    cv2.imwrite(real_image_path, dummy_image)
    print(f"Demo image created at {real_image_path}\n")

try:
    # Load the real handwritten image
    real_image = cv2.imread(real_image_path, cv2.IMREAD_GRAYSCALE)
    if real_image is None:
        raise FileNotFoundError(f"Could not load image at {real_image_path}")

    # Resize to 28x28
    real_image_resized = cv2.resize(real_image, (28, 28), interpolation=cv2.INTER_AREA)
    # Normalize to 0-1 range
    real_image_normalized = real_image_resized.astype('float32') / 255.0

    # Apply preprocessing
    processed_real_image = apply_histogram_equalization(real_image_normalized)

    # Extract features
    real_sobel_feat = extract_sobel_features(processed_real_image)
    real_canny_feat = extract_canny_features(processed_real_image)
    real_lbp_feat = extract_lbp_features(processed_real_image)

    real_image_features = np.array([
        real_sobel_feat[0],
        real_sobel_feat[1],
        real_canny_feat[0],
        real_lbp_feat[0],
        np.max(real_lbp_feat)
    ]).reshape(1, -1)

    # Scale features
    real_image_features_scaled = scaler.transform(real_image_features)

    # Predict
    prediction = best_model.predict(real_image_features_scaled)[0]

    # Get confidence if available
    if hasattr(best_model, 'predict_proba'):
        proba = best_model.predict_proba(real_image_features_scaled)[0]
        confidence = proba[list(alphabet).index(prediction)] if prediction in alphabet else 0
    else:
        confidence = 0

    print(f"{'='*70}")
    print(f"PREDICTION ON REAL HANDWRITTEN IMAGE")
    print(f"{'='*70}")
    print(f"Predicted Character: {prediction}")
    print(f"Confidence: {confidence:.4f}")
    print(f"Model Used: {best_model_name}")
    print(f"{'='*70}")

    # Display the processed real image
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))

    axes[0].imshow(real_image_normalized, cmap='gray')
    axes[0].set_title("Original Image", fontsize=12, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(processed_real_image, cmap='gray')
    axes[1].set_title(f"Processed (Predicted: {prediction})", fontsize=12, fontweight='bold')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

except FileNotFoundError as e:
    print(f"Error: {e}. Please provide a handwritten test image at '{real_image_path}'.")
except Exception as e:
    print(f"An error occurred: {e}")

---
## Part 9: Summary and Analysis

Comprehensive analysis of results

In [ ]:
print("\n" + "="*70)
print("COMPREHENSIVE ANALYSIS SUMMARY")
print("="*70 + "\n")

print("1. DATASET STATISTICS:")
print(f"   - Total samples: {len(images)}")
print(f"   - Training samples: {len(X_train)}")
print(f"   - Test samples: {len(X_test)}")
print(f"   - Number of classes: 26 (A-Z)")
print(f"   - Image size: 28×28 pixels")
print(f"   - Source: EMNIST Letters Dataset (TensorFlow Datasets)\n")

print("2. PREPROCESSING STEPS:")
print(f"   Normalization: 0-255 → 0-1 range")
print(f"   Histogram Equalization: Applied")
print(f"   Train-Test Split: 80-20 stratified\n")

print("3. FEATURE ENGINEERING (5 Features Total):")
print(f"   Sobel Edge Detection (mean magnitude)")
print(f"   Sobel Edge Detection (std magnitude)")
print(f"   Canny Edge Detection (edge density)")
print(f"   LBP Texture (first histogram bin)")
print(f"   LBP Texture (max histogram value)")
print(f"   Feature Normalization: StandardScaler\n")

print("4. MODEL PERFORMANCE (Test Set):")
for model_name, scores in sorted(model_scores.items(), key=lambda x: x[1]['accuracy'], reverse=True):
    acc = scores['accuracy']
    f1 = scores['f1_score']
    marker = "" if model_name == best_model_name else "  "
    print(f"   {marker} {model_name:<25}: Accuracy={acc:.4f}, F1={f1:.4f}")

print(f"\n{'='*70}")

---
## Individual Student Contribution

